# S1_00_demo_LLM_summary_run_across_months
This notebook automates the LLM summary generation for the 2022 Adzuna dataset using a recursive retry logic to maximize data coverage.

Pipeline Overview
Standard Run (Months 01–12): Iterates through the full calendar year of 2022 to generate initial summaries.

First Retry (Month 13): Aggregates and re-processes the failures (approx. 10%) identified during the initial runs of Months 01–12.

Final Cleanup (Month 14): Targets and re-runs any remaining failures specifically from the Month 13 retry pass.

# Get sizes of original datasets:

In [1]:
import numpy as np
from pathlib import Path

BASE = Path("/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_1_LLM_summary")

sizes = {}

for p in sorted(BASE.rglob("adzuna_month*_npz/*.npz")):
    with np.load(p, allow_pickle=True) as d:
        n = len(d["id"])   # or any 1D array
        sizes[p.name] = n

sizes


{'adzuna_month01.npz': 2840180,
 'adzuna_month01_retry.npz': 181252,
 'adzuna_month02.npz': 2692671,
 'adzuna_month02_retry.npz': 462202,
 'adzuna_month03.npz': 2521202,
 'adzuna_month03_retry.npz': 216284,
 'adzuna_month04.npz': 2492751,
 'adzuna_month04_retry.npz': 165824,
 'adzuna_month05.npz': 2684650,
 'adzuna_month05_retry.npz': 136979,
 'adzuna_month06.npz': 2280484,
 'adzuna_month06_retry.npz': 286734,
 'adzuna_month07.npz': 2505414,
 'adzuna_month07_retry.npz': 321006,
 'adzuna_month08.npz': 2164579,
 'adzuna_month08_retry.npz': 249724,
 'adzuna_month09.npz': 2102864,
 'adzuna_month09_retry.npz': 241700,
 'adzuna_month10.npz': 2333824,
 'adzuna_month10_retry.npz': 153444,
 'adzuna_month11.npz': 2192908,
 'adzuna_month11_retry.npz': 151496,
 'adzuna_month12.npz': 1838220,
 'adzuna_month12_retry.npz': 133940,
 'adzuna_month13.npz': 2573657,
 'adzuna_month13_retry.npz': 22289,
 'adzuna_month14.npz': 22289,
 'adzuna_month14_retry.npz': 250}

Following cells are periodic loops on the SBATCH command that gen summary:

    For each month:
    Splits the month’s NPZ into 100 GPU shards.
    Submits one Slurm array job using up to 100 GPUs in parallel.
    Waits until all tasks for that month leave the queue.
    Logs final job states but does not stop on failure.
    Proceeds to the next month regardless of errors.
    You handle reconciliation and reruns later.

# Sbatch Loop Months 9, 10, 11 and 12

In [ ]:
import math
import subprocess
import re
import time
from pathlib import Path

PROJECT = Path("/projects/a5u/adu_dev/aisi-economy-index")
BASE = PROJECT / "aisi_economy_index/store/isambard/202601/adzuna_2022_all_months"
SBATCH_SCRIPT = PROJECT / "nbs/isambard/2026_01/adzuna_22_all_months/run_llm_extract.sbatch"

sizes = {
    "adzuna_month09.npz": 2102864,
    "adzuna_month10.npz": 2333824,
    "adzuna_month11.npz": 2192908,
    "adzuna_month12.npz": 1838220,
}

TASKS = 100
CONCURRENCY = 100
POLL_SECONDS = 120

print("START RUN (Q4 months 09-12 sequential)")

for fname in sorted(sizes.keys()):
    n_rows = int(sizes[fname])
    month = fname[:-4]
    npz_path = BASE / month / fname
    npz_path = BASE / f"{month}_npz" / fname

    if not npz_path.exists():
        raise FileNotFoundError(f"Missing NPZ: {npz_path}")

    shard = math.ceil(n_rows / TASKS)

    print("\n==============================")
    print("MONTH:", month)
    print("NPZ:", npz_path)
    print("ROWS:", n_rows)
    print("TASKS:", TASKS, "(array 0-99)")
    print("SHARD:", shard)
    print("CONCURRENCY:", CONCURRENCY)

    job_name = f"llama_extract_all_{month}"

    cmd = [
        "sbatch",
        f"--job-name={job_name}",
        f"--array=0-{TASKS-1}%{CONCURRENCY}",
        SBATCH_SCRIPT,
        month,
        str(n_rows),
        str(shard),
        str(npz_path),
    ]

    res = subprocess.run(cmd, capture_output=True, text=True)
    if res.returncode != 0:
        raise RuntimeError(
            f"sbatch failed for {month}\nCMD:\n{' '.join(cmd)}\nSTDOUT:\n{res.stdout}\nSTDERR:\n{res.stderr}"
        )

    m = re.search(r"Submitted batch job (\d+)", res.stdout)
    if not m:
        raise RuntimeError(f"Could not parse job id from sbatch output:\n{res.stdout}")
    job_id = m.group(1)

    print("SUBMITTED job_id:", job_id)
    print("LOG PREFIX will be:", job_name)

    # wait until job leaves the queue
    while True:
        out = subprocess.run(
            ["squeue", "-j", job_id, "--noheader"],
            capture_output=True,
            text=True,
        ).stdout
        if not out.strip():
            print("DONE month:", month, "job_id:", job_id)
            break
        time.sleep(POLL_SECONDS)

    # log final state (do not fail-fast)
    sacct = subprocess.run(
        ["sacct", "-j", job_id, "--format=JobID,State%20", "--noheader"],
        capture_output=True, text=True
    )
    if sacct.returncode == 0:
        print("[STATE]", month, "job", job_id)
        print(sacct.stdout.strip())
    else:
        print("WARN: sacct failed:", sacct.stderr.strip())

print("\nQ4 FINISHED")


START RUN (Q4 months 09-12 sequential)

MONTH: adzuna_month09
NPZ: /projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month09_npz/adzuna_month09.npz
ROWS: 2102864
TASKS: 100 (array 0-99)
SHARD: 21029
CONCURRENCY: 100
SUBMITTED job_id: 2006447
LOG PREFIX will be: llama_extract_all_adzuna_month09


# Sbatch Loop Month 7 and 8

In [ ]:
import math
import subprocess
import re
import time
from pathlib import Path

PROJECT = Path("/projects/a5u/adu_dev/aisi-economy-index")
BASE = PROJECT / "aisi_economy_index/store/isambard/202601/adzuna_2022_all_months"
SBATCH_SCRIPT = PROJECT / "nbs/isambard/2026_01/adzuna_22_all_months/run_llm_extract.sbatch"

sizes = {
     'adzuna_month07.npz': 2505414,
     'adzuna_month08.npz': 2164579,
}

TASKS = 100
CONCURRENCY = 100
POLL_SECONDS = 120

print("START RUN (Q3 Las months 07-08 sequential)")

for fname in sorted(sizes.keys()):
    n_rows = int(sizes[fname])
    month = fname[:-4]
    npz_path = BASE / month / fname
    npz_path = BASE / f"{month}_npz" / fname

    if not npz_path.exists():
        raise FileNotFoundError(f"Missing NPZ: {npz_path}")

    shard = math.ceil(n_rows / TASKS)

    print("\n==============================")
    print("MONTH:", month)
    print("NPZ:", npz_path)
    print("ROWS:", n_rows)
    print("TASKS:", TASKS, "(array 0-99)")
    print("SHARD:", shard)
    print("CONCURRENCY:", CONCURRENCY)

    job_name = f"llama_extract_all_{month}"

    cmd = [
        "sbatch",
        f"--job-name={job_name}",
        f"--array=0-{TASKS-1}%{CONCURRENCY}",
        SBATCH_SCRIPT,
        month,
        str(n_rows),
        str(shard),
        str(npz_path),
    ]

    res = subprocess.run(cmd, capture_output=True, text=True)
    if res.returncode != 0:
        raise RuntimeError(
            f"sbatch failed for {month}\nCMD:\n{' '.join(cmd)}\nSTDOUT:\n{res.stdout}\nSTDERR:\n{res.stderr}"
        )

    m = re.search(r"Submitted batch job (\d+)", res.stdout)
    if not m:
        raise RuntimeError(f"Could not parse job id from sbatch output:\n{res.stdout}")
    job_id = m.group(1)

    print("SUBMITTED job_id:", job_id)
    print("LOG PREFIX will be:", job_name)

    # wait until job leaves the queue
    while True:
        out = subprocess.run(
            ["squeue", "-j", job_id, "--noheader"],
            capture_output=True,
            text=True,
        ).stdout
        if not out.strip():
            print("DONE month:", month, "job_id:", job_id)
            break
        time.sleep(POLL_SECONDS)

    # log final state (do not fail-fast)
    sacct = subprocess.run(
        ["sacct", "-j", job_id, "--format=JobID,State%20", "--noheader"],
        capture_output=True, text=True
    )
    if sacct.returncode == 0:
        print("[STATE]", month, "job", job_id)
        print(sacct.stdout.strip())
    else:
        print("WARN: sacct failed:", sacct.stderr.strip())

print("\nQ3 Partial (AUG/JUL) FINISHED")


START RUN (Q4 months 09-12 sequential)

MONTH: adzuna_month07
NPZ: /projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month07_npz/adzuna_month07.npz
ROWS: 2505414
TASKS: 100 (array 0-99)
SHARD: 25055
CONCURRENCY: 100
SUBMITTED job_id: 2013924
LOG PREFIX will be: llama_extract_all_adzuna_month07


# Sbatch Loop Month 1,2,3,4

In [1]:
import math
import subprocess
import re
import time
from pathlib import Path

PROJECT = Path("/projects/a5u/adu_dev/aisi-economy-index")
BASE = PROJECT / "aisi_economy_index/store/isambard/202601/adzuna_2022_all_months"
SBATCH_SCRIPT = PROJECT / "nbs/isambard/2026_01/adzuna_22_all_months/run_llm_extract.sbatch"

sizes = {
     'adzuna_month01.npz': 2840180,
     'adzuna_month02.npz': 2692671,
     'adzuna_month03.npz': 2521202,
     'adzuna_month04.npz': 2492751,

}

TASKS = 100
CONCURRENCY = 100
POLL_SECONDS = 120

print("START RUN 1st Semester Las months 01-06 sequential)")

for fname in sorted(sizes.keys()):
    n_rows = int(sizes[fname])
    month = fname[:-4]
    npz_path = BASE / month / fname
    npz_path = BASE / f"{month}_npz" / fname

    if not npz_path.exists():
        raise FileNotFoundError(f"Missing NPZ: {npz_path}")

    shard = math.ceil(n_rows / TASKS)

    print("\n==============================")
    print("MONTH:", month)
    print("NPZ:", npz_path)
    print("ROWS:", n_rows)
    print("TASKS:", TASKS, "(array 0-99)")
    print("SHARD:", shard)
    print("CONCURRENCY:", CONCURRENCY)

    job_name = f"llama_extract_all_{month}"

    cmd = [
        "sbatch",
        f"--job-name={job_name}",
        f"--array=0-{TASKS-1}%{CONCURRENCY}",
        SBATCH_SCRIPT,
        month,
        str(n_rows),
        str(shard),
        str(npz_path),
    ]

    res = subprocess.run(cmd, capture_output=True, text=True)
    if res.returncode != 0:
        raise RuntimeError(
            f"sbatch failed for {month}\nCMD:\n{' '.join(cmd)}\nSTDOUT:\n{res.stdout}\nSTDERR:\n{res.stderr}"
        )

    m = re.search(r"Submitted batch job (\d+)", res.stdout)
    if not m:
        raise RuntimeError(f"Could not parse job id from sbatch output:\n{res.stdout}")
    job_id = m.group(1)

    print("SUBMITTED job_id:", job_id)
    print("LOG PREFIX will be:", job_name)

    # wait until job leaves the queue
    while True:
        out = subprocess.run(
            ["squeue", "-j", job_id, "--noheader"],
            capture_output=True,
            text=True,
        ).stdout
        if not out.strip():
            print("DONE month:", month, "job_id:", job_id)
            break
        time.sleep(POLL_SECONDS)

    # log final state (do not fail-fast)
    sacct = subprocess.run(
        ["sacct", "-j", job_id, "--format=JobID,State%20", "--noheader"],
        capture_output=True, text=True
    )
    if sacct.returncode == 0:
        print("[STATE]", month, "job", job_id)
        print(sacct.stdout.strip())
    else:
        print("WARN: sacct failed:", sacct.stderr.strip())

print("\n1st Semester Las months 01-06 sequential FINISHED")


START RUN 1st Semester Las months 01-06 sequential)

MONTH: adzuna_month01
NPZ: /projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month01_npz/adzuna_month01.npz
ROWS: 2840180
TASKS: 100 (array 0-99)
SHARD: 28402
CONCURRENCY: 100
SUBMITTED job_id: 2016071
LOG PREFIX will be: llama_extract_all_adzuna_month01
DONE month: adzuna_month01 job_id: 2016071
[STATE] adzuna_month01 job 2016071
2016071_0               COMPLETED 
2016071_0.b+            COMPLETED 
2016071_0.0             COMPLETED 
2016071_1               COMPLETED 
2016071_1.b+            COMPLETED 
2016071_1.0             COMPLETED 
2016071_2               COMPLETED 
2016071_2.b+            COMPLETED 
2016071_2.0             COMPLETED 
2016071_3               COMPLETED 
2016071_3.b+            COMPLETED 
2016071_3.0             COMPLETED 
2016071_4                  FAILED 
2016071_4.b+               FAILED 
2016071_4.0                FAILED 
2016071_5               COM

KeyboardInterrupt: 

# Sbatch Loop Month 5 and 6

In [2]:
import math
import subprocess
import re
import time
from pathlib import Path

PROJECT = Path("/projects/a5u/adu_dev/aisi-economy-index")
BASE = PROJECT / "aisi_economy_index/store/isambard/202601/adzuna_2022_all_months"
SBATCH_SCRIPT = PROJECT / "nbs/isambard/2026_01/adzuna_22_all_months/run_llm_extract.sbatch"

sizes = {
     'adzuna_month05.npz': 2684650,
     'adzuna_month06.npz': 2280484,
}

TASKS = 100
CONCURRENCY = 100
POLL_SECONDS = 120

print("START RUN 1st Semester Las months 05-06 sequential)")

for fname in sorted(sizes.keys()):
    n_rows = int(sizes[fname])
    month = fname[:-4]
    npz_path = BASE / month / fname
    npz_path = BASE / f"{month}_npz" / fname

    if not npz_path.exists():
        raise FileNotFoundError(f"Missing NPZ: {npz_path}")

    shard = math.ceil(n_rows / TASKS)

    print("\n==============================")
    print("MONTH:", month)
    print("NPZ:", npz_path)
    print("ROWS:", n_rows)
    print("TASKS:", TASKS, "(array 0-99)")
    print("SHARD:", shard)
    print("CONCURRENCY:", CONCURRENCY)

    job_name = f"llama_extract_all_{month}"

    cmd = [
        "sbatch",
        f"--job-name={job_name}",
        f"--array=0-{TASKS-1}%{CONCURRENCY}",
        SBATCH_SCRIPT,
        month,
        str(n_rows),
        str(shard),
        str(npz_path),
    ]

    res = subprocess.run(cmd, capture_output=True, text=True)
    if res.returncode != 0:
        raise RuntimeError(
            f"sbatch failed for {month}\nCMD:\n{' '.join(cmd)}\nSTDOUT:\n{res.stdout}\nSTDERR:\n{res.stderr}"
        )

    m = re.search(r"Submitted batch job (\d+)", res.stdout)
    if not m:
        raise RuntimeError(f"Could not parse job id from sbatch output:\n{res.stdout}")
    job_id = m.group(1)

    print("SUBMITTED job_id:", job_id)
    print("LOG PREFIX will be:", job_name)

    # wait until job leaves the queue
    while True:
        out = subprocess.run(
            ["squeue", "-j", job_id, "--noheader"],
            capture_output=True,
            text=True,
        ).stdout
        if not out.strip():
            print("DONE month:", month, "job_id:", job_id)
            break
        time.sleep(POLL_SECONDS)

    # log final state (do not fail-fast)
    sacct = subprocess.run(
        ["sacct", "-j", job_id, "--format=JobID,State%20", "--noheader"],
        capture_output=True, text=True
    )
    if sacct.returncode == 0:
        print("[STATE]", month, "job", job_id)
        print(sacct.stdout.strip())
    else:
        print("WARN: sacct failed:", sacct.stderr.strip())

print("\n1st Semester Last months 05-06 sequential FINISHED")


START RUN 1st Semester Las months 05-06 sequential)

MONTH: adzuna_month05
NPZ: /projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month05_npz/adzuna_month05.npz
ROWS: 2684650
TASKS: 100 (array 0-99)
SHARD: 26847
CONCURRENCY: 100
SUBMITTED job_id: 2025067
LOG PREFIX will be: llama_extract_all_adzuna_month05
DONE month: adzuna_month05 job_id: 2025067
[STATE] adzuna_month05 job 2025067
2025067_0               COMPLETED 
2025067_0.b+            COMPLETED 
2025067_0.0             COMPLETED 
2025067_1               COMPLETED 
2025067_1.b+            COMPLETED 
2025067_1.0             COMPLETED 
2025067_2               COMPLETED 
2025067_2.b+            COMPLETED 
2025067_2.0             COMPLETED 
2025067_3               COMPLETED 
2025067_3.b+            COMPLETED 
2025067_3.0             COMPLETED 
2025067_4               COMPLETED 
2025067_4.b+            COMPLETED 
2025067_4.0             COMPLETED 
2025067_5               COM

# Retries: Month 13 & Month 14 Logic
In the notebook S1_01_summary_retry_dataset.ipynb, we handle the "long tail" of failed LLM summaries through two specific retry datasets:

1. Month 13: The Primary Retry Bucket
Source: All failed summaries across the standard calendar year (January to December 2022).

Volume: Approximately 10% failure rate from the initial runs.

Purpose: This dataset aggregates every record that failed to generate a valid summary in the first pass.

2. Month 14: The Secondary Retry Bucket
Source: The remaining failures specifically from the Month 13 run.

Purpose: This acts as a final "cleanup" dataset. It targets the most stubborn records that failed even after the first retry attempt.

## Month 13 (retries from month 1 to 12)

In [3]:
import math
import subprocess
import re
import time
from pathlib import Path

PROJECT = Path("/projects/a5u/adu_dev/aisi-economy-index")
BASE = PROJECT / "aisi_economy_index/store/isambard/202601/adzuna_2022_all_months"
SBATCH_SCRIPT = PROJECT / "nbs/isambard/2026_01/adzuna_22_all_months/run_llm_extract.sbatch"

sizes = {
     'adzuna_month13.npz': 2573657,
}

TASKS = 100
CONCURRENCY = 100
POLL_SECONDS = 120

print("START RUN 1st RETRY month13)")

for fname in sorted(sizes.keys()):
    n_rows = int(sizes[fname])
    month = fname[:-4]
    npz_path = BASE / month / fname
    npz_path = BASE / f"{month}_npz" / fname

    if not npz_path.exists():
        raise FileNotFoundError(f"Missing NPZ: {npz_path}")

    shard = math.ceil(n_rows / TASKS)

    print("\n==============================")
    print("MONTH:", month)
    print("NPZ:", npz_path)
    print("ROWS:", n_rows)
    print("TASKS:", TASKS, "(array 0-99)")
    print("SHARD:", shard)
    print("CONCURRENCY:", CONCURRENCY)

    job_name = f"llama_extract_all_{month}"

    cmd = [
        "sbatch",
        f"--job-name={job_name}",
        f"--array=0-{TASKS-1}%{CONCURRENCY}",
        SBATCH_SCRIPT,
        month,
        str(n_rows),
        str(shard),
        str(npz_path),
    ]

    res = subprocess.run(cmd, capture_output=True, text=True)
    if res.returncode != 0:
        raise RuntimeError(
            f"sbatch failed for {month}\nCMD:\n{' '.join(cmd)}\nSTDOUT:\n{res.stdout}\nSTDERR:\n{res.stderr}"
        )

    m = re.search(r"Submitted batch job (\d+)", res.stdout)
    if not m:
        raise RuntimeError(f"Could not parse job id from sbatch output:\n{res.stdout}")
    job_id = m.group(1)

    print("SUBMITTED job_id:", job_id)
    print("LOG PREFIX will be:", job_name)

    # wait until job leaves the queue
    while True:
        out = subprocess.run(
            ["squeue", "-j", job_id, "--noheader"],
            capture_output=True,
            text=True,
        ).stdout
        if not out.strip():
            print("DONE month:", month, "job_id:", job_id)
            break
        time.sleep(POLL_SECONDS)

    # log final state (do not fail-fast)
    sacct = subprocess.run(
        ["sacct", "-j", job_id, "--format=JobID,State%20", "--noheader"],
        capture_output=True, text=True
    )
    if sacct.returncode == 0:
        print("[STATE]", month, "job", job_id)
        print(sacct.stdout.strip())
    else:
        print("WARN: sacct failed:", sacct.stderr.strip())

print("\n1st Retry Month13 FINISHED")


START RUN 1st RETRY month13)

MONTH: adzuna_month13
NPZ: /projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month13_npz/adzuna_month13.npz
ROWS: 2573657
TASKS: 100 (array 0-99)
SHARD: 25737
CONCURRENCY: 100
SUBMITTED job_id: 2036568
LOG PREFIX will be: llama_extract_all_adzuna_month13


KeyboardInterrupt: 

## Month 14 (retries from month 13)

In [4]:
import math
import subprocess
import re
import time
from pathlib import Path

PROJECT = Path("/projects/a5u/adu_dev/aisi-economy-index")
BASE = PROJECT / "aisi_economy_index/store/isambard/202601/adzuna_2022_all_months"
SBATCH_SCRIPT = PROJECT / "nbs/isambard/2026_01/adzuna_22_all_months/run_llm_extract.sbatch"

sizes = {
     'adzuna_month14.npz': 22289,
}

TASKS = 100
CONCURRENCY = 100
POLL_SECONDS = 120

print("START RUN 1st RETRY month13)")

for fname in sorted(sizes.keys()):
    n_rows = int(sizes[fname])
    month = fname[:-4]
    npz_path = BASE / month / fname
    npz_path = BASE / f"{month}_npz" / fname

    if not npz_path.exists():
        raise FileNotFoundError(f"Missing NPZ: {npz_path}")

    shard = math.ceil(n_rows / TASKS)

    print("\n==============================")
    print("MONTH:", month)
    print("NPZ:", npz_path)
    print("ROWS:", n_rows)
    print("TASKS:", TASKS, "(array 0-99)")
    print("SHARD:", shard)
    print("CONCURRENCY:", CONCURRENCY)

    job_name = f"llama_extract_all_{month}"

    cmd = [
        "sbatch",
        f"--job-name={job_name}",
        f"--array=0-{TASKS-1}%{CONCURRENCY}",
        SBATCH_SCRIPT,
        month,
        str(n_rows),
        str(shard),
        str(npz_path),
    ]

    res = subprocess.run(cmd, capture_output=True, text=True)
    if res.returncode != 0:
        raise RuntimeError(
            f"sbatch failed for {month}\nCMD:\n{' '.join(cmd)}\nSTDOUT:\n{res.stdout}\nSTDERR:\n{res.stderr}"
        )

    m = re.search(r"Submitted batch job (\d+)", res.stdout)
    if not m:
        raise RuntimeError(f"Could not parse job id from sbatch output:\n{res.stdout}")
    job_id = m.group(1)

    print("SUBMITTED job_id:", job_id)
    print("LOG PREFIX will be:", job_name)

    # wait until job leaves the queue
    while True:
        out = subprocess.run(
            ["squeue", "-j", job_id, "--noheader"],
            capture_output=True,
            text=True,
        ).stdout
        if not out.strip():
            print("DONE month:", month, "job_id:", job_id)
            break
        time.sleep(POLL_SECONDS)

    # log final state (do not fail-fast)
    sacct = subprocess.run(
        ["sacct", "-j", job_id, "--format=JobID,State%20", "--noheader"],
        capture_output=True, text=True
    )
    if sacct.returncode == 0:
        print("[STATE]", month, "job", job_id)
        print(sacct.stdout.strip())
    else:
        print("WARN: sacct failed:", sacct.stderr.strip())

print("\n1st Retry Month13 FINISHED")


START RUN 1st RETRY month13)

MONTH: adzuna_month14
NPZ: /projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/isambard/202601/adzuna_2022_all_months/adzuna_month14_npz/adzuna_month14.npz
ROWS: 22289
TASKS: 100 (array 0-99)
SHARD: 223
CONCURRENCY: 100
SUBMITTED job_id: 2037529
LOG PREFIX will be: llama_extract_all_adzuna_month14
DONE month: adzuna_month14 job_id: 2037529
[STATE] adzuna_month14 job 2037529
2037529_0               COMPLETED 
2037529_0.b+            COMPLETED 
2037529_0.0             COMPLETED 
2037529_1               COMPLETED 
2037529_1.b+            COMPLETED 
2037529_1.0             COMPLETED 
2037529_2               COMPLETED 
2037529_2.b+            COMPLETED 
2037529_2.0             COMPLETED 
2037529_3               COMPLETED 
2037529_3.b+            COMPLETED 
2037529_3.0             COMPLETED 
2037529_4               COMPLETED 
2037529_4.b+            COMPLETED 
2037529_4.0             COMPLETED 
2037529_5               COMPLETED 
2037529_5.b+       